### AI Recruiting Advisor
##### Use this bot to ask for advice on your portfolio. By putting in a website, you'll get insights into what's working on your portfolio, what isn't, and what careers you are the most well positioned for.

In [1]:
# Load Packages
import os
import sys
import asyncio
import subprocess
import requests
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display
from openai import OpenAI

NOTEBOOK_DIR = os.path.dirname(os.path.abspath(globals().get("__vsc_ipynb_file__", os.getcwd())))
SCRAPER_SCRIPT = os.path.join(NOTEBOOK_DIR, "scraper.py")

In [2]:
## Prompting

# Scraper

# Some websites need you to use proper headers when fetching them:
headers = {
 "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

def _fetch_rendered_html_sync(url):
    """Fetch a page's fully-rendered, unprocessed HTML by running scraper.py in raw mode."""
    result = subprocess.run(
        [sys.executable, SCRAPER_SCRIPT, url, "--mode", "raw"],
        capture_output=True, text=True, encoding="utf-8", check=True,
    )
    return result.stdout

async def _fetch_website_content(url):
    """Run the subprocess."""
    return await asyncio.to_thread(_fetch_rendered_html_sync, url)

class Website:

    def __init__(self, url, html):
        """
        Create this Website object from already-fetched HTML using the BeautifulSoup library.
        Use the async `Website.create(url)` factory below instead of calling this directly.
        """
        self.url = url
        soup = BeautifulSoup(html, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        self.text = soup.body.get_text(separator="\n", strip=True) if soup.body else ""

    @classmethod
    async def create(cls, url):
        """
        Async factory: fetch the page with requests, then fall back to a headless-browser
        render (via _fetch_rendered_html) if the page looks JS-rendered (empty title/body).
        """
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.content, 'html.parser')
        title = soup.title.string if soup.title else None
        body_text = soup.body.get_text(separator="\n", strip=True) if soup.body else ""

        html = response.content
        if not title or len(body_text) < 50:
            html = await _fetch_website_content(url)

        return cls(url, html)

# User Prompt Generator

def user_prompt_generator(website):
    user_prompt = f"You are looking at a website titled {website.title}."
    user_prompt += f"The website content is as follows."
    user_prompt += website.text
    return user_prompt

# System Prompt

system_prompt = "You are a recruiting support assistant that analyzes the content of a personal portfolio website and provides insight into\
                what's working well, what isn't, and what career positions and roles the candidate is well positioned for. Follow with advice on how the user can further\
                improve their portfolio to support their goals. Start by trying to understand what roles the user is gunning for through their website, conduct research into\
                what stands out for those roles, and then start providing insights and advice. Do not hallucinate; base every insight strictly on the content of the website."

# Messages

def messages_prompt(system_prompt, user_prompt):
    messages = [{"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}]
    return messages

# Full

async def get_prompt_from_url(website_url):
    website = await Website.create(website_url)
    user_prompt = user_prompt_generator(website)
    messages = messages_prompt(system_prompt, user_prompt)

    return messages

In [3]:
# Final Function

async def portfolio_reviewer(url):
    openai = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

    messages = await get_prompt_from_url(url)
    response = openai.chat.completions.create(model="llama3.2", messages=messages)
    summary = response.choices[0].message.content
    display(Markdown(summary))
    return summary

In [6]:
# Testing

await portfolio_reviewer("https://www.reneesingh.com")

Based on the content of Renee Singh's personal portfolio website, I've attempted to understand what roles she is targeting and what stands out for those roles.

Renee seems to be positioning herself as a data scientist with expertise in machine learning, data engineering, business intelligence, and finance. Her experience in working on projects related to resource discovery platforms, financial modeling, and real-time portfolio analytics suggests that she may be interested in roles at the intersection of technology and business.

Here are some insights and advice:

**Insights:**

1. **Interdisciplinary background**: Renee's background in data science and economics is evident throughout her website. This combination of technical depth and business acumen may make her a strong candidate for roles that require both technical expertise and business awareness.
2. **Project management experience**: Her experience as Director of the hackathon, Project Management role at AMARA, and Teaching Assistant at UW Information School demonstrates her ability to lead projects and manage teams. This skillset is valuable in many industries, particularly in tech and finance.
3. **Technical skills**: Renee's proficiency in languages like Python, R, JavaScript, and TypeScript, as well as tools like Tableau, PowerBI, and Airflow, suggests that she has a strong technical foundation.

**Advice:**

1. **Emphasize relevance to target roles**: While Renee's projects are impressive, it's essential to highlight the specific skills and experience directly relevant to her target roles (e.g., data scientist at a fintech company or business intelligence specialist).
2. **Showcase success metrics**: Providing concrete metrics and outcomes from her projects can help quantify her achievements and demonstrate the impact she can make in a role.
3. **Tailor online presence for recruitment**: Review and optimize Renee's website to ensure it is easily discoverable by recruiters and hiring managers. Consider adding keywords, categorizing projects, or creating separate sections for each relevant skill set.
4. **Expand on expertise areas**: While Renee has demonstrated expertise in machine learning and data engineering, expanding on related skills like model explanation, interpretability, or explainable AI might complement her experience and make her a more competitive candidate.

To help Renee improve her portfolio, I suggest focusing on the following:

1. **Curate relevant projects**: Prioritize showcasing 2-3 key projects that demonstrate your expertise in areas directly applicable to target roles.
2. **Quantify achievements**: Gather metrics or statistics from each project and include them in the project descriptions.
3. **Develop targeted writing samples**: Write summaries of notable research papers, articles, or essays related to her research interests (e.g., using HAISP dataset) to provide insight into her thought leadership and expertise.
4. **Create an easily navigable website structure**: Organize Renee's website content in a logical and accessible manner to facilitate easy discovery by recruiters.

By implementing these suggestions, Renee can strengthen her portfolio and effectively communicate her value proposition to potential employers.

"Based on the content of Renee Singh's personal portfolio website, I've attempted to understand what roles she is targeting and what stands out for those roles.\n\nRenee seems to be positioning herself as a data scientist with expertise in machine learning, data engineering, business intelligence, and finance. Her experience in working on projects related to resource discovery platforms, financial modeling, and real-time portfolio analytics suggests that she may be interested in roles at the intersection of technology and business.\n\nHere are some insights and advice:\n\n**Insights:**\n\n1. **Interdisciplinary background**: Renee's background in data science and economics is evident throughout her website. This combination of technical depth and business acumen may make her a strong candidate for roles that require both technical expertise and business awareness.\n2. **Project management experience**: Her experience as Director of the hackathon, Project Management role at AMARA, and T